# 1. Instalasi Library dan Import

In [ ]:
!pip install transformers torch scikit-learn pandas numpy sastrawi

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE

# 2. Load Dataset

In [ ]:
from google.colab import files
uploaded = files.upload()

df = pd.read_csv('nlp.csv')
print(df.shape)
print(df.head())
print(df.columns.tolist())

# 3. Pelabelan Sentimen menggunakan IndoBERT

In [ ]:
from transformers import pipeline

# Load model
sentiment_pipeline = pipeline(
    "text-classification",
    model="w11wo/indonesian-roberta-base-sentiment-classifier"
)

# Fungsi labeling
def get_sentiment(text):
    try:
        result = sentiment_pipeline(str(text)[:512])[0]
        label = result['label'].lower()
        if label == 'positive':
            return 1
        elif label == 'negative':
            return -1
        else:
            return 0
    except:
        return 0

# Labeling semua data (ini akan butuh beberapa menit)
print("Sedang melabeli data...")
df['sentiment'] = df['full_text'].apply(get_sentiment)
print("Selesai!")
print(df['sentiment'].value_counts())

# 4. Feature Extraction (TF-IDF) dan Split Data

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE

# TF-IDF
tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=5000)
X = tfidf.fit_transform(df['text_clean'])
y = df['sentiment']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Distribusi train:", pd.Series(y_train).value_counts())

# 5. Oversampling dengan SMOTE dan Pelatihan Model Naive Bayes

In [ ]:
# SMOTE hanya pada training data
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("Distribusi setelah SMOTE:")
print(pd.Series(y_train_resampled).value_counts())

# Model Naive Bayes
nb_model = MultinomialNB(alpha=0.1)
nb_model.fit(X_train_resampled, y_train_resampled)

# Evaluasi pada test data
y_pred = nb_model.predict(X_test)

print("\n=== EVALUASI PADA TEST DATA ===")
print(classification_report(y_test, y_pred, target_names=['Negative', 'Neutral', 'Positive']))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# 6. Visualisasi Hasil

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Distribusi label sebelum dan sesudah SMOTE
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

label_names = {-1: 'Negative', 0: 'Neutral', 1: 'Positive'}
order = ['Negative', 'Neutral', 'Positive']
colors = ['#e74c3c', '#95a5a6', '#2ecc71']

# Sebelum SMOTE
before = pd.Series(y_train).map(label_names).value_counts().reindex(order)
axes[0].bar(order, before.values, color=colors)
axes[0].set_title('Distribusi Label\nSebelum SMOTE')
axes[0].set_ylabel('Jumlah Data')

# Sesudah SMOTE
after = pd.Series(y_train_resampled).map(label_names).value_counts().reindex(order)
axes[1].bar(order, after.values, color=colors)
axes[1].set_title('Distribusi Label\nSetelah SMOTE')
axes[1].set_ylabel('Jumlah Data')

plt.tight_layout()
plt.show()

# 2. Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative', 'Neutral', 'Positive'],
            yticklabels=['Negative', 'Neutral', 'Positive'])
plt.title('Confusion Matrix — Test Data')
plt.ylabel('Aktual')
plt.xlabel('Prediksi')
plt.show()

# 7. Evaluasi F1-Score per Kelas

In [ ]:
# 3. F1-Score per kelas
from sklearn.metrics import classification_report
import pandas as pd

report = classification_report(y_test, y_pred,
                                target_names=['Negative', 'Neutral', 'Positive'],
                                output_dict=True)

f1_scores = {
    'Negative': report['Negative']['f1-score'],
    'Neutral': report['Neutral']['f1-score'],
    'Positive': report['Positive']['f1-score']
}

plt.figure(figsize=(7, 4))
bars = plt.bar(f1_scores.keys(), f1_scores.values(),
               color=['#e74c3c', '#95a5a6', '#2ecc71'])

# Tambah angka di atas bar
for bar, val in zip(bars, f1_scores.values()):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.2f}', ha='center', fontsize=11)

plt.title('F1-Score per Kelas Sentimen')
plt.ylabel('F1-Score')
plt.ylim(0, 1)
plt.axhline(y=0.7, color='gray', linestyle='--', alpha=0.5, label='Threshold 0.7')
plt.legend()
plt.show()

#1. Membandingan dengan model SVM

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Model SVM
svm_model = LinearSVC(random_state=42, max_iter=1000)
svm_model.fit(X_train_resampled, y_train_resampled)

# Evaluasi pada test data
y_pred_svm = svm_model.predict(X_test)

print("=== EVALUASI SVM PADA TEST DATA ===")
print(classification_report(y_test, y_pred_svm,
                            target_names=['Negative', 'Neutral', 'Positive']))

# Confusion Matrix SVM
cm_svm = confusion_matrix(y_test, y_pred_svm)
plt.figure(figsize=(7, 5))
sns.heatmap(cm_svm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Negative', 'Neutral', 'Positive'],
            yticklabels=['Negative', 'Neutral', 'Positive'])
plt.title('Confusion Matrix SVM — Test Data')
plt.ylabel('Aktual')
plt.xlabel('Prediksi')
plt.show()

#2. Visualisasi Perbandingan Naive Bayes vs SVM

In [ ]:
categories = ['Negative', 'Neutral', 'Positive', 'Accuracy']

nb_scores = [
    report['Negative']['f1-score'],
    report['Neutral']['f1-score'],
    report['Positive']['f1-score'],
    report['accuracy']
]

report_svm = classification_report(y_test, y_pred_svm,
                                    target_names=['Negative', 'Neutral', 'Positive'],
                                    output_dict=True)

svm_scores = [
    report_svm['Negative']['f1-score'],
    report_svm['Neutral']['f1-score'],
    report_svm['Positive']['f1-score'],
    report_svm['accuracy']
]

x = np.arange(len(categories))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, nb_scores, width, label='Naive Bayes', color='#3498db')
bars2 = ax.bar(x + width/2, svm_scores, width, label='SVM', color='#e67e22')

# Angka di atas bar
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.2f}', ha='center', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.2f}', ha='center', fontsize=9)

ax.set_title('Perbandingan Naive Bayes vs SVM')
ax.set_ylabel('Score')
ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.set_ylim(0, 1.1)
ax.legend()
ax.axhline(y=0.7, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()